In [2]:
import networkx as nx
import numpy as np
from sklearn.model_selection import StratifiedKFold
import ot
import pandas as pd
import time
import random
from gwgraphs import *
import pyreadr

In [3]:
#### GW EXPERIMENTS WITH COBRE DATA ####
## Set up data ##
nets = pyreadr.read_r('networks.rds')
labels = pyreadr.read_r('labels.rds')
df = nets[None]
labels_df = pyreadr.read_r('labels.rds')[None]
df_full = df.copy()
df_full['labels'] = labels_df.values
df_full

,0,1,2,3,4,5,6,7,8,9,...,34444,34445,34446,34447,34448,34449,34450,34451,34452,labels
0,0.937964,0.036858,0.233517,0.357248,0.300294,0.072664,0.131153,0.095430,-0.006198,0.153983,...,0.429179,0.346439,0.111311,0.347075,0.284673,0.135142,0.419904,0.235776,0.486115,1.0
1,0.122836,0.197231,-0.159027,0.100587,-0.087402,-0.066368,-0.055579,0.404727,-0.103306,0.056261,...,0.620209,0.406304,0.381617,0.472145,0.447971,0.134374,0.205255,0.508898,0.492664,1.0
2,0.609382,0.175494,-0.024736,0.170654,0.051464,0.185128,0.184114,-0.046622,0.348225,0.265189,...,0.189252,0.131152,0.122525,0.354497,0.190312,-0.247709,0.293514,-0.170872,0.343746,1.0
3,0.150549,-0.065422,0.246946,0.060092,0.004350,0.402564,-0.125902,0.027463,0.349015,0.171557,...,0.743368,0.299041,-0.069612,0.211402,0.352202,-0.040681,0.270711,0.239744,0.166712,1.0
4,0.818724,0.466267,0.460847,0.608540,0.632230,0.188454,0.303265,0.236847,0.442380,0.218805,...,0.356244,0.081661,0.125469,-0.032464,0.291471,-0.098654,0.611725,0.204744,0.380010,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,0.677140,0.321552,0.117745,0.168275,0.211885,0.099639,0.313155,0.182389,0.224607,-0.130002,...,0.648144,0.544748,0.145445,0.572884,0.718308,0.506452,0.704904,0.631571,0.091547,1.0
120,0.551664,0.178585,0.051788,-0.264080,-0.038901,0.025136,0.188339,-0.131339,0.492654,-0.280235,...,0.488505,-0.069349,0.008380,0.019600,0.146450,-0.144245,0.292833,-0.130710,0.055620,-1.0
121,0.603361,0.581785,0.363042,0.190121,0.184959,0.046118,0.057942,0.106672,0.346750,0.056851,...,0.360031,0.565816,0.255710,0.349328,0.486091,0.039027,0.167118,-0.259183,0.173690,1.0
122,0.251397,0.441282,0.190952,0.485988,0.200581,0.364325,0.431289,0.082483,0.536304,0.200708,...,0.147478,0.551669,0.189728,0.287138,0.172694,0.092271,-0.081608,0.316341,0.171579,-1.0


In [4]:
unhealthy = df_full[df_full['labels'] == 1]
healthy = df_full[df_full['labels'] == -1]
X = np.append(unhealthy, healthy, axis=0)  # Combine unhealthy and healthy
y = [1] * len(unhealthy) + [-1] * len(healthy)
y = np.array(y)  # Convert y into a numpy array

In [4]:
num_folds = 5
n = 263

k_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
best_accuracy = 0
best_k = 0
all_accuracies = {}

skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
start = time.time()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

    X_train, X_test = df_full.iloc[train_index], df_full.iloc[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train = X_train.drop(columns='labels')
    X_test = X_test.drop(columns='labels')

    # Convert each row into upper triangular matrices and store them in a list
    train_graph = []
    test_graph = []

    for idx, row in X_train.iterrows():
        linmat = np.array(row)
        mat = np.zeros((n, n))
        mat[np.triu_indices(n, 1)] = linmat
        mat += mat.T
        train_graph.append(mat)
    print("Training data converted")

    for idx, row in X_test.iterrows():
        linmat = np.array(row)
        mat = np.zeros((n, n))
        mat[np.triu_indices(n, 1)] = linmat
        mat += mat.T
        test_graph.append(mat)
    print("Testing data converted")

    # Compute all pairwise distances
    all_distances = []
    for test in test_graph:
        test_distances = []
        for train, train_label in zip(train_graph, y_train):
            gw, log = ot.gromov.gromov_wasserstein(test,
                                                   train,
                                                   loss_fun='square_loss',
                                                   log=True,
                                                   random_seed=42)
            test_distances.append((log['gw_dist'], train_label))
        all_distances.append(test_distances)

    # Classify for each k
    for k in k_list:
        correct_count = 0
        for distances, ground_truth_label in zip(all_distances, y_test):
            # Sort based on distances
            sorted_distances = sorted(distances, key=lambda x: x[0])
            # Take the k nearest neighbors
            k_nearest_neighbors = sorted_distances[:k]
            # Determine the most common label
            labels_counts = {}
            for _, label in k_nearest_neighbors:
                labels_counts[label] = labels_counts.get(label, 0) + 1
            predicted_label = max(labels_counts, key=labels_counts.get)

            if predicted_label == ground_truth_label:
                correct_count += 1

        # Calculate sensitivity, specificity, classification accuracy
        tp = fp = tn = fn = 0
        for distances, ground_truth_label in zip(all_distances, y_test):
            sorted_distances = sorted(distances, key=lambda x: x[0])
            k_nearest_neighbors = sorted_distances[:k]
            labels_counts = {}
            for _, label in k_nearest_neighbors:
                labels_counts[label] = labels_counts.get(label, 0) + 1
            predicted_label = max(labels_counts, key=labels_counts.get)
            
            if predicted_label == 1 and ground_truth_label == 1:
                tp += 1
            elif predicted_label == 1 and ground_truth_label == -1:
                fp += 1
            elif predicted_label == -1 and ground_truth_label == -1:
                tn += 1
            else:  # predicted_label == -1 and ground_truth_label == 1
                fn += 1

        classification_accuracy = (correct_count / len(X_test)) * 100
        sensitivity = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
        specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0
        
        print(f"Fold {i}, k={k}:")
        print(f"  Classification Accuracy: {classification_accuracy:.2f}%")
        print(f"  Sensitivity: {sensitivity:.2f}%")
        print(f"  Specificity: {specificity:.2f}%")

        # Update metrics
        if k not in all_accuracies:
            all_accuracies[k] = {'acc': [], 'sens': [], 'spec': []}
        all_accuracies[k]['acc'].append(classification_accuracy)
        all_accuracies[k]['sens'].append(sensitivity)
        all_accuracies[k]['spec'].append(specificity)

end = time.time()
print("Total runtime, seconds:", end - start)

# Calculate and print final results
for k in k_list:
    avg_accuracy = np.mean(all_accuracies[k]['acc'])
    avg_sensitivity = np.mean(all_accuracies[k]['sens'])
    avg_specificity = np.mean(all_accuracies[k]['spec'])
    
    print(f"\nResults for k={k}:")
    print(f"  Average Accuracy: {avg_accuracy:.2f}%")
    print(f"  Average Sensitivity: {avg_sensitivity:.2f}%")
    print(f"  Average Specificity: {avg_specificity:.2f}%")
    
    if avg_accuracy > best_accuracy:
        best_accuracy = avg_accuracy
        best_k = k

print(f"\nBest accuracy: {best_accuracy:.2f}% achieved with k={best_k}")

Fold 0:
  Train: index=[  0   1   2   3   4   6   7   9  10  12  14  16  17  18  19  20  21  23
  24  25  26  27  29  30  31  32  33  34  36  38  39  40  41  42  44  45
  46  47  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63  64
  65  66  68  69  70  72  73  74  76  78  79  82  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 101 102 103 104 105 106 107 108 110
 112 113 114 115 117 118 120 121 123]
  Test:  index=[  5   8  11  13  15  22  28  35  37  43  48  67  71  75  77  80  81  83
  84 100 109 111 116 119 122]
Training data converted
Testing data converted
Fold 0, k=1:
  Classification Accuracy: 56.00%
  Sensitivity: 0.00%
  Specificity: 100.00%
Fold 0, k=2:
  Classification Accuracy: 56.00%
  Sensitivity: 0.00%
  Specificity: 100.00%
Fold 0, k=3:
  Classification Accuracy: 56.00%
  Sensitivity: 0.00%
  Specificity: 100.00%
Fold 0, k=4:
  Classification Accuracy: 56.00%
  Sensitivity: 0.00%
  Specificity: 100.00%
Fold 0, k=5:
  Classification Accuracy: 56.00

In [89]:
### gLGW kNN ###
## CITATION ###
# https://github.com/Gorgotha/LGW #
# lgw_procedure, LGW_graph, and LGW_eucl are taken directly from this code.
# k_folds_glgw_knn is a modified function from the above code. Everything else is my work.
def lgw_procedure(M_ref,
                  height_ref,
                  posns,
                  Ms,
                  heights,
                  max_iter=1000,
                  mode="euclidean"):
  """
  Computes the generalized linear Gromov-Wasserstein distance between similarity matrices.
  Parameters:
  - M_ref: reference barycenter
  - height_ref: distribution/weights in the source space
  - posns: If mode="euclidean", posns is a list of positions to compute the Euclidean embeddings. If mode="graph", enter None.
  - Ms: list of similarity matrices (a matrix of measures)
  - heights: distribution/weights in the target space
  - max_iter: maximum number of iterations (default: 1000)
  - mode: "euclidean" or "graph" (default: "euclidean"). Determines which norm to use for the distance computation. Choice of norm depends on type of data.
  Output:
  - lgw: a matrix containing the pairwise generalized linear Gromov-Wasserstein distances
  - et-st: total time
  """
  assert mode in ["euclidean", "graph"]
  N = len(Ms)

  Ps = []  #GW Plans
  Ts = []  #barycentric projections
  st = time.time()
  for i in range(0, N):
    #GW computation
    P = ot.gromov.gromov_wasserstein(M_ref,
                                     Ms[i],
                                     height_ref,
                                     heights[i],
                                     "square_loss",
                                     log=True)[0]
    Ps.append(P)

    #euclidean barycentric projection
    if mode == "euclidean":
      T = (np.divide(P.T, height_ref).T).dot(posns[i])
    #generalized barycentric projection
    else:
      T = []
      k = len(Ms[i])
      k_ref = len(M_ref)
      for v in range(k_ref):
        barycentricity = []
        weights = P[v] / height_ref[v]
        for w in range(k):
          breakBool = False
          if weights[w] == 1:
            bary = w
            breakBool = True
            break
          barycentricity.append(weights.dot(Ms[i][w]**2))
        if breakBool:
          T.append(bary)
        else:
          bary = np.argmin(barycentricity)
          T.append(bary)
    Ts.append(T)

  #LGW computation
  lgw = np.zeros((N, N))
  for i in range(N):
    for j in range(i + 1, N):
      if mode == "euclidean":
        lgw[i, j] = LGW_eucl(Ts[i], Ts[j], height_ref)
      else:
        lgw[i, j] = LGW_graph(Ts[i], Ts[j], Ms[i], Ms[j], height_ref)
  lgw += lgw.T
  et = time.time()
  return lgw, et - st


"""
The paper: https://arxiv.org/pdf/2112.11964
The relevant sections on the barycenter computations are sections III C and IV C. The difference between the two functions ocurrs due to differences in how the barycentric projections are computed, depending on the type of data you have.
"""


def LGW_graph(T1, T2, D1, D2, sigma):
  """
  Works directly with the graph distance matrices D1, D2 and uses the transport maps T1, T2 to select relevant entries from these matrices. It then computes the distance between graphs using their matrix representations.
  """
  return np.sqrt(
      np.sum(
          np.multiply((D1[T1].T[T1].T - D2[T2].T[T2].T)**2,
                      np.outer(sigma, sigma))))


def LGW_eucl(T1, T2, sigma, normalized=False):
  """
  First converts the transport maps T1, T2 into Euclidean distance matrices using ot.dist, with an optional normalization. Then it compares these Euclidean distances. In lgw_procedure, the "euclidean" mode will additionally require position data to compute Euclidean embeddings.
  """
  M1 = ot.dist(T1, T1, metric="euclidean")
  M2 = ot.dist(T2, T2, metric="euclidean")
  if normalized:
    M1 = M1 / np.max(M1)
    M2 = M2 / np.max(M2)
  return np.sqrt(np.sum(np.multiply((M1 - M2)**2, np.outer(sigma, sigma))))


def glgw_knn(lgw_matrix,
             labels,
             train_index,
             test_index,
             k_list,
             log_knn=False):
    """
        Computes the k-nearest neighbors for a given graph using the LGW distance.
        Parameters:
        - lgw_matrix: the LGW distance matrix containing all the pairwise LGW distances between graphs
        - labels: the labels of the graphs
        - train_index: the indices of the graphs from the training set
        - test_index: the indices of the graphs from the test set
        - k_list: list of k values to evaluate
        - log_knn: whether to print the classification accuracy log (default: False)
        Output:
        - classification accuracy: the classification accuracy of the kNN algorithm
    """
    class_acc_dict = {k: [] for k in k_list}
    metrics_dict = {k: {'sensitivity': [], 'specificity': []} for k in k_list}

    X_train, y_train = lgw_matrix[train_index], [labels[idx] for idx in train_index]
    X_test, y_test = lgw_matrix[test_index], [labels[idx] for idx in test_index]

    # Initialize counters for each k
    correct_counts = {k: 0 for k in k_list}
    confusion_matrix = {k: {'TP': 0, 'TN': 0, 'FP': 0, 'FN': 0} for k in k_list}

    for test_idx, ground_truth_label in zip(test_index, y_test):
        distances = [(lgw_matrix[test_idx, idx], y_train[j])
                     for j, idx in enumerate(train_index)]
        distances.sort(key=lambda x: x[0])

        for k in k_list:
          k_nearest_neighbors = distances[:k]
          labels_counts = {}
          for _, label in k_nearest_neighbors:
            labels_counts[label] = labels_counts.get(label, 0) + 1
          predicted_label = max(labels_counts, key=labels_counts.get)
    
          if ground_truth_label == 1:
            if predicted_label == 1:
              confusion_matrix[k]['TP'] += 1
            else:
              confusion_matrix[k]['FN'] += 1
          else:
            if predicted_label == -1:
              confusion_matrix[k]['TN'] += 1
            else:
              confusion_matrix[k]['FP'] += 1
    
          if predicted_label == ground_truth_label:
            correct_counts[k] += 1

    # Calculate accuracies after all predictions
    for k in k_list:
        # Overall accuracy
        classification_accuracy = (correct_counts[k] / len(test_index)) * 100
        class_acc_dict[k].append(classification_accuracy)

        # Calculate sensitivity and specificity
        sensitivity = (confusion_matrix[k]['TP'] / 
                  (confusion_matrix[k]['TP'] + confusion_matrix[k]['FN']) * 100 
                  if (confusion_matrix[k]['TP'] + confusion_matrix[k]['FN']) > 0 else 0)
    
        specificity = (confusion_matrix[k]['TN'] / 
                  (confusion_matrix[k]['TN'] + confusion_matrix[k]['FP']) * 100
                  if (confusion_matrix[k]['TN'] + confusion_matrix[k]['FP']) > 0 else 0)

        metrics_dict[k]['sensitivity'].append(sensitivity)
        metrics_dict[k]['specificity'].append(specificity)

        if log_knn:
          print(f"k={k}, Sensitivity: {sensitivity}%")
          print(f"k={k}, Specificity: {specificity}%")
          print(f"k={k}, Overall Classification Accuracy: {classification_accuracy}%")

    return class_acc_dict, metrics_dict


def k_folds_glgw_knn(X, y, n, k_bary, Ms, heights, num_folds, seed, k_knn_list):
    """
    Implements k-fold cross-validation in conjunction with the linearized kNN algorithm.
    Parameters:
    - X: the input data
    - y: the labels of the data
    - k_bary: the number of points to use for the reference barycenter in the LGW computation
    - Ms: the list of similarity matrices (a matrix of measures)
    - heights: the distribution/weights in the target space
    - num_folds: the number of folds to use for cross-validation
    - seed: the seed for the random number generator
    - k_knn_list: the list of k values to use for k-fold cross-validation
    """
    np.random.seed(seed)
    k_folds = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)
    results = {k: [] for k in k_knn_list}
    classes = [-1, 1]
    for train_index, test_index in k_folds.split(X, y):
        X_train, X_test = df_full.iloc[train_index], df_full.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]
    
        X_train = X_train.drop(columns='labels')
        X_test = X_test.drop(columns='labels')
    
        # Convert each row into upper triangular matrices and store them in a list
        train_graph = []
        test_graph = []
        Ms_processed = []
    
        for idx, row in X_train.iterrows():
            linmat = np.array(row)
            mat = np.zeros((n, n))
            mat[np.triu_indices(n, 1)] = linmat
            mat += mat.T
            train_graph.append(mat)
        print("Training data converted")
    
        for idx, row in X_test.iterrows():
            linmat = np.array(row)
            mat = np.zeros((n, n))
            mat[np.triu_indices(n, 1)] = linmat
            mat += mat.T
            test_graph.append(mat)
        print("Testing data converted")

        for mat in Ms:
            if isinstance(mat, (list, np.ndarray)):
                linmat = np.array(mat)
            else:
                linmat = np.array(mat.values)
            
            mat = np.zeros((n, n))
            mat[np.triu_indices(n, 1)] = linmat
            mat += mat.T
            Ms_processed.append(mat)
        
        # Compute reference barycenter for the training set
        
        idx_bary = []
        for i in classes:
            # Find the indices of instances with label i
            indices = np.where(y_train == i)[0]
            if indices.size > 0:
                idx_bary.extend(
                    np.random.choice(train_index[indices],
                                     size=min(5, len(indices)),
                                     replace=False))
                
        bary_start_time = time.time()
        M_ref = ot.gromov.gromov_barycenters(k_bary,
                                             Cs=np.array(Ms_processed)[idx_bary],
                                             ps=np.array(heights)[idx_bary],
                                             p=ot.unif(k_bary),
                                             lambdas=ot.unif(len(idx_bary)),
                                             loss_fun='square_loss',
                                             max_iter=200,
                                             tol=1e-12,
                                             random_state=seed)
        bary_computation_time = time.time() - bary_start_time
        print(f"Time taken for reference barycenter computation: {bary_computation_time:.4f} seconds")
        
        height_ref = ot.unif(k_bary)
        lgw_matrix, lgw_time = lgw_procedure(M_ref,
                                             height_ref,
                                             None,
                                             Ms_processed,
                                             heights,
                                             mode="graph")

        print(f"Time taken for LGW computation: {lgw_time:.4f} seconds")

        class_acc_dict, by_class_acc_dict = glgw_knn(lgw_matrix,
                              y,
                              train_index,
                              test_index, k_knn_list, log_knn=True)
    # Find best k and print stats
    best_k = k_knn_list[0]
    best_acc = 0
    k_avg_accuracies = {}

    for k in k_knn_list:
       avg_acc = sum(class_acc_dict[k]) / len(class_acc_dict[k])
       k_avg_accuracies[k] = {
        'overall': avg_acc,
        'sensitivity': sum(by_class_acc_dict[k]['sensitivity']) / len(by_class_acc_dict[k]['sensitivity']),
        'specificity': sum(by_class_acc_dict[k]['specificity']) / len(by_class_acc_dict[k]['specificity'])
       }

       print(f"\nk={k} Average Overall Accuracy: {avg_acc}%")
       print(f"k={k} Average Sensitivity: {k_avg_accuracies[k]['sensitivity']}%")
       print(f"k={k} Average Specificity: {k_avg_accuracies[k]['specificity']}%")

       if avg_acc > best_acc:
         best_acc = avg_acc
         best_k = k

    print(f"\nBest performance achieved with k={best_k}:")
    print(f"Overall Accuracy: {k_avg_accuracies[best_k]['overall']}%")
    print(f"Sensitivity: {k_avg_accuracies[best_k]['sensitivity']}%")
    print(f"Specificity: {k_avg_accuracies[best_k]['specificity']}%")

In [90]:
Ms = []
for matrix in X:
    matrix = np.delete(matrix, -1)
    Ms.append(np.abs(matrix))

In [91]:
k_bary = 263
n = 263
heights = [
    np.full(263, 1 / 263) for graph in X
]
num_folds = 5
k_knn_list=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [92]:
k_folds_glgw_knn(X=X,
            y=y,
            n=n,
            k_bary=k_bary,
            Ms=Ms,
            heights=heights,
            num_folds=num_folds,
            seed=42,
            k_knn_list=k_knn_list)

Training data converted
Testing data converted
Time taken for reference barycenter computation: 149.7458 seconds
Time taken for LGW computation: 15.4081 seconds
k=1, Sensitivity: 9.090909090909092%
k=1, Specificity: 92.85714285714286%
k=1, Overall Classification Accuracy: 56.00000000000001%
k=2, Sensitivity: 9.090909090909092%
k=2, Specificity: 92.85714285714286%
k=2, Overall Classification Accuracy: 56.00000000000001%
k=3, Sensitivity: 54.54545454545454%
k=3, Specificity: 42.857142857142854%
k=3, Overall Classification Accuracy: 48.0%
k=4, Sensitivity: 45.45454545454545%
k=4, Specificity: 64.28571428571429%
k=4, Overall Classification Accuracy: 56.00000000000001%
k=5, Sensitivity: 81.81818181818183%
k=5, Specificity: 21.428571428571427%
k=5, Overall Classification Accuracy: 48.0%
k=6, Sensitivity: 36.36363636363637%
k=6, Specificity: 42.857142857142854%
k=6, Overall Classification Accuracy: 40.0%
k=7, Sensitivity: 81.81818181818183%
k=7, Specificity: 21.428571428571427%
k=7, Overall C